# Tweets Sentiment (Basic)

Notebook ini untuk klasifikasi sentimen dasar berbasis data tweet dari Excel Anda.

## Sumber data
- File: 08-Text Mining/tweets_clf.xlsx
- Sheet training: training
- Sheet prediksi: test

## Target belajar
1. Membaca data teks dari Excel
2. Cleaning teks sederhana
3. Membuat fitur TF-IDF
4. Melatih model klasifikasi sederhana
5. Evaluasi akurasi dasar
6. Prediksi data baru dari sheet test

## 1) Install dan import library

In [ ]:
# Uncomment jika library belum ada
# !pip install pandas openpyxl scikit-learn matplotlib seaborn

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

## 2) Baca data training dari Excel

In [ ]:
path_file = "tweets_clf.xlsx"
sheet_train = "training"

try:
    df = pd.read_excel(path_file, sheet_name=sheet_train)
except ValueError:
    print(f"Sheet '{sheet_train}' tidak ditemukan. Menggunakan sheet pertama.")
    df = pd.read_excel(path_file)

print("Ukuran data training:", df.shape)
df.head()

## 3) Deteksi kolom teks dan label
Jika otomatis tidak cocok, silakan set manual text_col dan label_col.

In [ ]:
kandidat_text = ["tweet", "text", "content", "ulasan", "komentar"]
kandidat_label = ["label", "sentiment", "sentimen", "kelas", "target"]

lower_map = {c.lower(): c for c in df.columns}

text_col = next((lower_map[k] for k in kandidat_text if k in lower_map), None)
label_col = next((lower_map[k] for k in kandidat_label if k in lower_map), None)

print("Kolom terdeteksi -> text:", text_col, "| label:", label_col)
print("Daftar kolom:", df.columns.tolist())

## 3b) EDA ringkas sebelum cleaning
Kita lihat kualitas data awal: nilai kosong, duplikasi, distribusi label, dan panjang tweet.

In [ ]:
print("=== Info Data Awal ===")
print("Jumlah baris:", len(df))
print("Jumlah duplikasi baris:", df.duplicated().sum())
print("Missing value per kolom:")
print(df.isna().sum())

if text_col is None or label_col is None:
    print("\nEDA label/teks dilewati karena text_col atau label_col belum terdeteksi.")
else:
    eda_df = df.dropna(subset=[text_col, label_col]).copy()
    eda_df["text_len"] = eda_df[text_col].astype(str).str.len()
    
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    label_count = eda_df[label_col].value_counts()
    sns.barplot(x=label_count.index.astype(str), y=label_count.values, ax=axes[0], palette="Set2")
    axes[0].set_title("Distribusi Label Sentimen")
    axes[0].set_xlabel("Label")
    axes[0].set_ylabel("Jumlah")
    axes[0].tick_params(axis="x", rotation=25)

    sns.histplot(eda_df["text_len"], bins=20, kde=True, ax=axes[1], color="#2a9d8f")
    axes[1].set_title("Distribusi Panjang Tweet")
    axes[1].set_xlabel("Jumlah Karakter")
    axes[1].set_ylabel("Frekuensi")

    plt.tight_layout()
    plt.show()

    print("\nContoh 3 tweet terpendek:")
    display(eda_df[[text_col, label_col, "text_len"]].sort_values("text_len").head(3))
    print("\nContoh 3 tweet terpanjang:")
    display(eda_df[[text_col, label_col, "text_len"]].sort_values("text_len", ascending=False).head(3))

## 4) Cleaning teks sederhana

In [ ]:
def clean_text(s):
    s = str(s).lower()
    s = re.sub(r"http\S+|www\S+", " ", s)
    s = re.sub(r"@\w+", " ", s)
    s = re.sub(r"#", " ", s)
    s = re.sub(r"[^a-zA-Z\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

df_work = df.copy()
df_work = df_work.dropna(subset=[text_col, label_col])
df_work["clean_text"] = df_work[text_col].apply(clean_text)

df_work[[text_col, "clean_text", label_col]].head()

## 5) Split data train-test
Kita pakai 80% data untuk training, 20% untuk testing.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df_work["clean_text"],
    df_work[label_col],
    test_size=0.2,
    random_state=42,
    stratify=df_work[label_col]
)

print("Jumlah train:", len(X_train))
print("Jumlah test:", len(X_test))

## 6) TF-IDF + Naive Bayes
Ini baseline yang ringan dan mudah dipahami pemula.

In [ ]:
vectorizer = TfidfVectorizer(max_features=2000, ngram_range=(1, 1))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

## 7) Evaluasi model

In [ ]:
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("Accuracy:", round(acc, 4))
print("\nConfusion Matrix:")
print(cm)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

plt.figure(figsize=(6, 4))
labels_cm = sorted(y_test.astype(str).unique())
sns.heatmap(cm, annot=True, fmt="d", cmap="YlGnBu", xticklabels=labels_cm, yticklabels=labels_cm)
plt.title("Heatmap Confusion Matrix")
plt.xlabel("Prediksi")
plt.ylabel("Aktual")
plt.tight_layout()
plt.show()

## 8) Uji 1 kalimat baru

In [ ]:
contoh_tweet = "pelayanan hari ini sangat cepat dan memuaskan"
contoh_bersih = clean_text(contoh_tweet)
contoh_vec = vectorizer.transform([contoh_bersih])
prediksi = model.predict(contoh_vec)[0]

print("Tweet:", contoh_tweet)
print("Prediksi sentimen:", prediksi)

## 9) Prediksi sentimen untuk data sheet test

In [ ]:
sheet_test = "test"

try:
    df_test = pd.read_excel(path_file, sheet_name=sheet_test)
except ValueError:
    raise ValueError(f"Sheet '{sheet_test}' tidak ditemukan pada file {path_file}")

if text_col not in df_test.columns:
    raise ValueError(
        f"Kolom teks '{text_col}' tidak ada di sheet test. Kolom tersedia: {df_test.columns.tolist()}"
    )

df_test_work = df_test.copy()
df_test_work = df_test_work.dropna(subset=[text_col])
df_test_work["clean_text"] = df_test_work[text_col].apply(clean_text)

X_test_sheet = vectorizer.transform(df_test_work["clean_text"])
df_test_work["prediksi_sentimen"] = model.predict(X_test_sheet)

output_file = "tweets_sentiment_test_result.csv"
df_test_work.to_csv(output_file, index=False)

print("Ukuran data test:", df_test.shape)
print("Data test setelah drop NA kolom teks:", df_test_work.shape)
print("Hasil prediksi sheet test tersimpan di:", output_file)
display(df_test_work[[text_col, "prediksi_sentimen"]].head(10))

# Visualisasi distribusi prediksi sentimen pada data test
pred_count = df_test_work["prediksi_sentimen"].value_counts()
pred_pct = (pred_count / pred_count.sum() * 100).round(2)

plt.figure(figsize=(7, 4))
ax = sns.barplot(x=pred_count.index.astype(str), y=pred_count.values, palette="Set2")
plt.title("Distribusi Prediksi Sentimen (Data Test)")
plt.xlabel("Label Prediksi")
plt.ylabel("Jumlah")

for i, (count, pct) in enumerate(zip(pred_count.values, pred_pct.values)):
    ax.text(i, count, f"{count} ({pct}%)", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()